# レッスン13 - エージェントメモリー


## セットアップ

このノートブックでは、**Microsoft Agent Framework**（MAF）を使用して、**永続的なメモリ**を持つ旅行予約エージェントを構築する方法を示します。

エージェントの作業メモリ、短期メモリ、長期メモリといったさまざまなタイプのメモリが、会話を通じてエージェントが情報を保持し利用する方法にどのように影響するかを学びます。

**前提条件:**
- チャットモデルがデプロイされた Azure AI Foundry プロジェクト（例：`gpt-4o-mini`）。
- Azure CLI でログイン済みであること — ターミナルで `az login` を実行してください。
- `AZURE_AI_PROJECT_ENDPOINT` — あなたの Azure AI Foundry プロジェクトのエンドポイント。
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — デプロイ済みモデルの名前。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## エージェントのメモリの種類

AIエージェントはさまざまな種類のメモリを活用できます。それぞれ異なる目的を持っています。

### ワーキングメモリ
会話スレッド自体—単一セッションで交換されるメッセージ。エージェントは同じスレッド内の以前のメッセージを参照して一貫性を保つことができます。MAFでは、**`agent.create_session()`** により作成され、`AgentSession`を返します。

### 短期記憶
タスクやセッションの期間中持続する情報ですが、永久的に保存されるわけではありません。例えば、エージェントはマルチターンプランニングの会話中に事実を蓄積し、それらを利用して最終的な旅程を作成することがあります。

### 長期記憶
**セッションをまたいで**持続する好みや事実。再び訪問するユーザーは、食事制限や旅行スタイルを繰り返す必要がありません。長期記憶は通常、外部のストア—データベース、ファイル、ベクターインデックス—によってサポートされ、ツールを通じてエージェントに提供されます。


## セッションによる作業メモリ

最も単純なメモリの形態は会話セッションです。同じセッション（`agent.create_session()`で作成されたもの）を連続する`agent.run()`呼び出しに渡すと、エージェントはその会話の全履歴を参照でき、以前の詳細を思い出すことができます。

旅行代理店を作成して、作業メモリを実演しましょう。


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

エージェントが予算を正しく思い出せたのは、両方のメッセージが同じセッションを共有しているためです。これは**作業記憶**です — セッションの寿命の間だけ存在します。

### 新しいスレッドでは何が起こるか？

**新しい**セッションを作成すると、エージェントは前の会話の記憶を持っていません：


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## 長期記憶パターン

ユーザーの好みを**セッションを超えて**記憶するには、会話スレッドの外側に存在する永続的なストアが必要です。エージェントはこのストアに、情報を保存・取得するために呼び出せる関数である**ツール**を通じてアクセスします。

以下では、単純なインメモリの好みストアを実装します（本番環境ではデータベースやベクターインデックスを使用します）そして、それをエージェントが使えるツールとして公開します。

### アーキテクチャ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### シナリオ1 — 初めてのユーザーが記念日旅行を予約する

サラは初めて訪れます。エージェントはツールを使って彼女の好みを保存し、それを基にホテルを推薦すべきです。


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### シナリオ 2 — 数週間後にサラが戻ってくる場合

サラは**まったく新しいスレッド**を開始します（新しいセッションをシミュレート）。作業メモリは空ですが、長期の好みストアには彼女の情報がまだあります。エージェントはそれを取得し、パーソナライズされたおすすめに使用する必要があります。


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

このレッスンでは、3種類のエージェントメモリと、それらをMicrosoft Agent Frameworkで実装する方法について学びました。

| メモリタイプ | MAF メカニズム | 有効期間 |
|---|---|---|
| **ワーキング** | `agent.create_session()` | 単一の会話 |
| **短期** | スレッド内に蓄積されたコンテキスト | 単一のタスク / セッション |
| **長期** | `@tool` 関数を通じてアクセスされる外部ストア | セッションを跨ぐ |

### 主なポイント
1. **`agent.create_session()`** はワーキングメモリを提供します — セッション内でエージェントは会話履歴全体を参照できます。
2. **新しいセッションはコンテキストを失います** — 長期メモリがなければエージェントは過去の会話を思い出せません。
3. **`@tool` 関数がギャップを埋めます** — これによりエージェントは永続的なストアに情報を保存・取得できます。
4. **パーソナライゼーションは時間とともに向上します** — もっと多くの好みが蓄積されるほど、エージェントの推奨は改善されます。

### 実際の応用例
- **カスタマーサービス**: 顧客の履歴や好みを記憶する
- **パーソナルアシスタント**: 数日または数週間にわたるコンテキストを維持する
- **ヘルスケア**: 患者情報や好みを追跡する
- **Eコマース**: 履歴に基づいたパーソナライズされたショッピング

### 次のステップ
- インメモリの辞書をデータベースやベクターストア（例：Azure AI Search）に置き換える
- 時間に敏感な情報のためにメモリの有効期限を追加する
- 共有メモリを持つマルチエージェントシステムを構築する
- 知識グラフに基づくメモリを備えたCogneeノートブックを探求する


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「[Co-op Translator](https://github.com/Azure/co-op-translator)」を使用して翻訳されました。正確性を目指しておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご理解ください。原文は常に正式な情報源としてご確認ください。重要な内容については、専門の人間翻訳をお勧めします。本翻訳の利用により生じた誤解や誤訳について、当方は一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
